# Decision Trees and Ensemble Learning

Welcome to the laboratory for dicision tress and tree-based ensemble algorithms.

## 6.1 Credit Risk Scoring Project

Dataset: https://github.com/gastonstat/CreditScoring

    

In [1]:
# Environment setup

import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

# Data source

data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-06-trees/CreditScoring.csv'

In [2]:
# Load Data into environment

!wget $data

--2026-10-30 09:16:40--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-06-trees/CreditScoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 182489 (178K) [text/plain]
Saving to: ‘CreditScoring.csv’

CreditScoring.csv   100%[===================>] 178.21K  --.-KB/s    in 0.003s  

2026-10-30 09:16:40 (50.4 MB/s) - ‘CreditScoring.csv’ saved [182489/182489]



In [8]:
df = pd.read_csv(data)
df.head()

,Status,Seniority,Home,Time,Age,Marital,Records,Job,Expenses,Income,Assets,Debt,Amount,Price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


In [9]:
# Prepare the data for ML processing

df.columns = df.columns.str.lower()

### Re-encoding the categorical variables

The columns `status`, `home`, `marital`, `records` and job are categorical, but in the file they are encoded as numbers. For example, status is 1, 2 or 0. That's not very intuitive, so we decode them into strings.

In [11]:
df.status.value_counts()

status
1    3200
2    1254
0       1
Name: count, dtype: int64

1 = paid back the loan
2 = defaulted

We will use pd.map() to first translate the numbers into strings()

In [13]:
status_values = {
    1: 'ok',
    2: 'default',
    0: 'unk'
}

df.status = df.status.map(status_values)

Usw.

In [14]:
home_values = {
    1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'
}

df.home = df.home.map(home_values)

marital_values = {
    1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'
}

df.marital = df.marital.map(marital_values)

records_values = {
    1: 'no',
    2: 'yes',
    0: 'unk'
}

df.records = df.records.map(records_values)

job_values = {
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
}

df.job = df.job.map(job_values)

In [15]:
df.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,ok,9,rent,60,30,married,no,freelance,73,129,0,0,800,846
1,ok,17,rent,60,58,widow,no,fixed,48,131,0,0,1000,1658
2,default,10,owner,36,46,married,yes,freelance,90,200,3000,0,2000,2985
3,ok,0,rent,60,24,single,no,fixed,63,182,2500,0,900,1325
4,ok,0,rent,36,26,single,no,fixed,46,107,0,0,310,910


### Missing Values

"Not every problem shows up as a missing value. Let's look at the summary statistics:"

In [16]:
df.describe().round()

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0
mean,8.0,46.0,37.0,56.0,763317.0,1060341.0,404382.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,8703625.0,10217569.0,6344253.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1118.0
50%,5.0,48.0,36.0,51.0,120.0,3500.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,166.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,99999999.0,99999999.0,99999999.0,5000.0,11140.0


The maximum values of `income`, `assets` and `debt` look suspicious: 99999999. This dataset encodes missing values as a long number of nines. Instead of NaN, "unknown" became 99999999. We replace these values with NaN:

In [18]:
for c in ['income', 'assets', 'debt']:
    df[c] = df[c].replace(to_replace=99999999, value=np.nan)

In [19]:
df.describe().round()

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4455.0,4455.0,4455.0,4455.0,4421.0,4408.0,4437.0,4455.0,4455.0
mean,8.0,46.0,37.0,56.0,131.0,5403.0,343.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,86.0,11573.0,1246.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1118.0
50%,5.0,48.0,36.0,51.0,120.0,3000.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,165.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,959.0,300000.0,30000.0,5000.0,11140.0


### Removing the One Record with a Status of Unknown

We also have that single customer with status unk. Since we only care about the two classes - ok and default - we remove this row:

In [21]:
# Make a new dataframe from the old dataframe using every record, so long as the status is not 'unk'. 
# Then reset the dataframe index

df = df[df.status != 'unk'].reset_index(drop=True)

In [22]:
df.count()

status       4454
seniority    4454
home         4454
time         4454
age          4454
marital      4454
records      4454
job          4454
expenses     4454
income       4420
assets       4407
debt         4436
amount       4454
price        4454
dtype: int64

### Split source data in to Train / Validation / Test

Now we split the data. As in the previous modules, we do it in two steps: first we split off the test set (20%), then we split the rest into train and validation (75/25 of what remains, which gives us the 60/20/20 distribution):

In [23]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=11)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=11)

In [24]:
# We reset the indexes so they are fresh and unique in each dataframe:

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

### Preparing the Target Variable

The models give us a probability, so the target needs to be a number. We encode `status`: 0 for `ok`, 1 for `default`:

In [26]:
y_train = (df_train.status == 'default').astype('int').values
y_val = (df_val.status == 'default').astype('int').values
y_test = (df_test.status == 'default').astype('int').values

Finally, we delete status from the dataframes - otherwise we would use the target as a feature, and the model would learn to cheat:

In [27]:
del df_train['status']
del df_val['status']
del df_test['status']

## Decision Trees

We train our first decision tree with scikit-learn on the credit scoring data, see what overfitting looks like, and learn how to control the size of a tree. The following function is an algorithm that _could_ be used to perform credit scoring.

In [28]:
def assess_risk(client):
    if client['records'] == 'yes':
        if client['job'] == 'parttime':
            return 'default'
        else:
            return 'ok'
    else:
        if client['assets'] > 6000:
            return 'ok'
        else:
            return 'default'

In [30]:
xi = df_train.iloc[0].to_dict()
assess_risk(xi)

'ok'